# Pod Service Reuse Analysis
## Baseline vs Transformation Service Reuse (S4)

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams.update({
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.dpi': 150,
    'axes.grid': True,
    'grid.alpha': 0.3
})

BASE_DIR = Path('~/Desktop/paper_experiments').expanduser()

EXPERIMENTS = {
    'Baseline U2':    BASE_DIR / 'baseline_sequential' / 'nextflow_rnaseq_partial_resume_2026_05_28_06_56_27_df3720c9',
    'Baseline U4':    BASE_DIR / 'baseline_sequential' / 'nextflow_rnaseq_partial_resume_2026_05_28_08_23_15_380543a9',
    'Pod Service U2': BASE_DIR / 'pod_service' / 'nextflow_rnaseq_partial_resume_2026_05_29_08_49_05_177a4c52',
    'Pod Service U4': BASE_DIR / 'pod_service' / 'nextflow_rnaseq_partial_resume_2026_05_29_09_47_52_7125d12a',
}

def load_csv(folder, name):
    path = Path(folder) / 'data' / name
    return pd.read_csv(path) if path.exists() else None

def load_run_json(folder):
    p = Path(folder) / 'run.json'
    return json.loads(p.read_text()) if p.exists() else None

def get_treatment(df):
    if 'treatment' in df.columns:
        return df[df['treatment'] == 'Treatment'].copy()
    return df.copy()

def get_energy(folder):
    df = load_csv(folder, 'kepler_node_cpu_power_sum.csv')
    if df is not None:
        return get_treatment(df)['kepler_node_cpu_power_sum'].sum() * 10
    return 0

def get_duration(folder):
    rj = load_run_json(folder)
    return rj.get('treatment_duration') if rj else None

tasks_per_user = 548

# Pre-computed values
data = {
    'Baseline U2':    {'energy_kj': 214.99, 'duration_min': 85.78,  'throughput': 12.78, 'users': 2},
    'Baseline U4':    {'energy_kj': 432.18, 'duration_min': 172.89, 'throughput': 12.68, 'users': 4},
    'Pod Service U2': {'energy_kj': 182.41, 'duration_min': 57.78,  'throughput': 18.97, 'users': 2},
    'Pod Service U4': {'energy_kj': 365.87, 'duration_min': 115.60, 'throughput': 18.96, 'users': 4},
}

scenarios  = ['Baseline', 'Pod Service']
keys_u2 = ['Baseline U2', 'Pod Service U2']
keys_u4 = ['Baseline U4', 'Pod Service U4']
x = np.arange(len(scenarios))
width = 0.35
colors_u2 = ['#1f77b4', '#6baed6']
colors_u4 = ['#d62728', '#ef6548']

print('Setup done')

## Plot 1 — Total Energy Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

energy_u2 = [data[k]['energy_kj'] for k in keys_u2]
energy_u4 = [data[k]['energy_kj'] for k in keys_u4]

bars1 = ax.bar(x - width/2, energy_u2, width, label='U2 (2 users)', color=colors_u2, edgecolor='black', linewidth=0.5)
bars2 = ax.bar(x + width/2, energy_u4, width, label='U4 (4 users)', color=colors_u4, edgecolor='black', linewidth=0.5)

for bar in list(bars1) + list(bars2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., h * 1.01, f'{h:.1f} kJ', ha='center', va='bottom', fontsize=10)

# Annotate savings
ax.annotate('', xy=(x[1] - width/2, energy_u2[1]), xytext=(x[0] - width/2, energy_u2[0]),
            arrowprops=dict(arrowstyle='->', color='#1f77b4', lw=1.5))
ax.text(x[0] + 0.05, (energy_u2[0] + energy_u2[1])/2, '-15.2%', color='#1f77b4', fontsize=10, fontweight='bold')
ax.annotate('', xy=(x[1] + width/2, energy_u4[1]), xytext=(x[0] + width/2, energy_u4[0]),
            arrowprops=dict(arrowstyle='->', color='#d62728', lw=1.5))
ax.text(x[0] + 0.55, (energy_u4[0] + energy_u4[1])/2, '-15.3%', color='#d62728', fontsize=10, fontweight='bold')

ax.set_xlabel('Scenario')
ax.set_ylabel('Total Energy Consumption (kJ)')
ax.set_title('Total Energy Consumption:\nBaseline vs Transformation Service Reuse')
ax.set_xticks(x)
ax.set_xticklabels(scenarios)
ax.legend(title='User Count')
plt.tight_layout()
plt.savefig('pod_service_energy.pdf', bbox_inches='tight')
plt.savefig('pod_service_energy.png', bbox_inches='tight')
plt.show()
print('Saved: pod_service_energy.pdf')

## Plot 2 — Pipeline Duration

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

dur_u2 = [data[k]['duration_min'] for k in keys_u2]
dur_u4 = [data[k]['duration_min'] for k in keys_u4]

bars1 = ax.bar(x - width/2, dur_u2, width, label='U2 (2 users)', color=colors_u2, edgecolor='black', linewidth=0.5)
bars2 = ax.bar(x + width/2, dur_u4, width, label='U4 (4 users)', color=colors_u4, edgecolor='black', linewidth=0.5)

for bar in list(bars1) + list(bars2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., h * 1.01, f'{h:.1f} min', ha='center', va='bottom', fontsize=10)

ax.annotate('', xy=(x[1] - width/2, dur_u2[1]), xytext=(x[0] - width/2, dur_u2[0]),
            arrowprops=dict(arrowstyle='->', color='#1f77b4', lw=1.5))
ax.text(x[0] + 0.05, (dur_u2[0] + dur_u2[1])/2, '-32.6%', color='#1f77b4', fontsize=10, fontweight='bold')
ax.annotate('', xy=(x[1] + width/2, dur_u4[1]), xytext=(x[0] + width/2, dur_u4[0]),
            arrowprops=dict(arrowstyle='->', color='#d62728', lw=1.5))
ax.text(x[0] + 0.55, (dur_u4[0] + dur_u4[1])/2, '-33.1%', color='#d62728', fontsize=10, fontweight='bold')

ax.set_xlabel('Scenario')
ax.set_ylabel('Total Duration (min)')
ax.set_title('Total Pipeline Duration:\nBaseline vs Transformation Service Reuse')
ax.set_xticks(x)
ax.set_xticklabels(scenarios)
ax.legend(title='User Count')
plt.tight_layout()
plt.savefig('pod_service_duration.pdf', bbox_inches='tight')
plt.savefig('pod_service_duration.png', bbox_inches='tight')
plt.show()
print('Saved: pod_service_duration.pdf')

## Plot 3 — System Throughput

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

tp_u2 = [data[k]['throughput'] for k in keys_u2]
tp_u4 = [data[k]['throughput'] for k in keys_u4]

bars1 = ax.bar(x - width/2, tp_u2, width, label='U2 (2 users)', color=colors_u2, edgecolor='black', linewidth=0.5)
bars2 = ax.bar(x + width/2, tp_u4, width, label='U4 (4 users)', color=colors_u4, edgecolor='black', linewidth=0.5)

for bar in list(bars1) + list(bars2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., h * 1.01, f'{h:.1f}', ha='center', va='bottom', fontsize=10)

ax.annotate('', xy=(x[1] - width/2, tp_u2[1]), xytext=(x[0] - width/2, tp_u2[0]),
            arrowprops=dict(arrowstyle='->', color='#1f77b4', lw=1.5))
ax.text(x[0] + 0.05, (tp_u2[0] + tp_u2[1])/2, '+48.5%', color='#1f77b4', fontsize=10, fontweight='bold')
ax.annotate('', xy=(x[1] + width/2, tp_u4[1]), xytext=(x[0] + width/2, tp_u4[0]),
            arrowprops=dict(arrowstyle='->', color='#d62728', lw=1.5))
ax.text(x[0] + 0.55, (tp_u4[0] + tp_u4[1])/2, '+49.6%', color='#d62728', fontsize=10, fontweight='bold')

ax.set_xlabel('Scenario')
ax.set_ylabel('Throughput (tasks/min)')
ax.set_title('System Throughput:\nBaseline vs Transformation Service Reuse')
ax.set_xticks(x)
ax.set_xticklabels(scenarios)
ax.legend(title='User Count')
plt.tight_layout()
plt.savefig('pod_service_throughput.pdf', bbox_inches='tight')
plt.savefig('pod_service_throughput.png', bbox_inches='tight')
plt.show()
print('Saved: pod_service_throughput.pdf')

## Plot 4 — Combined Savings Summary

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

metrics = ['Energy\nSaving', 'Duration\nSaving', 'Throughput\nGain']
savings_u2 = [15.2, 32.6, 48.5]
savings_u4 = [15.3, 33.1, 49.6]

x3 = np.arange(len(metrics))
bars1 = ax.bar(x3 - width/2, savings_u2, width, label='U2 (2 users)', color='#6baed6', edgecolor='black', linewidth=0.5)
bars2 = ax.bar(x3 + width/2, savings_u4, width, label='U4 (4 users)', color='#ef6548', edgecolor='black', linewidth=0.5)

for bar in list(bars1) + list(bars2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., h * 1.01, f'{h:.1f}%', ha='center', va='bottom', fontsize=10)

ax.axhline(y=0, color='black', linewidth=0.8)
ax.set_xlabel('Metric')
ax.set_ylabel('Improvement vs Baseline (%)')
ax.set_title('Performance Improvements:\nTransformation Service Reuse vs Baseline')
ax.set_xticks(x3)
ax.set_xticklabels(metrics)
ax.legend(title='User Count')
plt.tight_layout()
plt.savefig('pod_service_savings_summary.pdf', bbox_inches='tight')
plt.savefig('pod_service_savings_summary.png', bbox_inches='tight')
plt.show()
print('Saved: pod_service_savings_summary.pdf')